In [28]:
import numpy as np
import pandas as pd

In [171]:
redcap_table = pd.read_csv('/mnt/leif/littlab/users/aguilac/CNTSurgicalRepositor-CarlosUpdatedOutcome_DATA_LABELS_2024-09-04_1518.csv', index_col = 0)
pt_list = pd.read_csv('/mnt/leif/littlab/users/aguilac/Interictal_Spike_Analysis/HUMAN/working_feat_extract_code/5-propagation/dataset/ML_data/MUSC/pooled_spearman_all.csv', index_col =0)

pt_list = pt_list.dropna(subset = ['pt_id'])
pt_list['pt_id'] = pt_list['pt_id'].astype(int)
redcap_table = redcap_table.dropna(subset = ['HUP Number'])
redcap_table['HUP Number']=redcap_table['HUP Number'].astype(int)



In [172]:
merged_table = redcap_table.merge(pt_list, left_on='HUP Number', right_on='pt_id')

In [173]:
merged_table.columns

Index(['HUP Number', 'Intervention', 'Type of surgical procedure',
       'Resection laterality:  left or right', 'Resection lobe:',
       'Ablation target:', 'Specify other ablation target:',
       'Months since surgery at time of follow up #1',
       'Follow Up #1 Status (9-15 months from surgery):',
       'ILAE outcome category (past 12 months) for resection and laser patients:',
       'Engel Outcome Classification for resection and laser patients ',
       'Months since surgery at time of follow up #2',
       'ILAE outcome category (past 12 months) for resection and laser patients:.1',
       'Engel Outcome Classification for resection and laser patients .1',
       'Sex assigned at birth', 'MRI Lesion Type:', 'Other (MRI lesion type):',
       'Age at seizure onset?',
       'Age of seizure onset if no age number provided (e.g. early childhood):',
       'Age at Implant', 'Implant bilateral or unilateral:', 'spike_rate_corr',
       'SOZ', 'pt_id', 'rise_amp_corr', 'decay_am

## Basic demographic info for HUP

In [55]:
merged_table['Sex assigned at birth'].value_counts()

Sex assigned at birth
Male      27
Female    24
Name: count, dtype: int64

In [56]:
merged_table['Age at seizure onset?'].max()

48.0

In [58]:
print(len(merged_table['Age at Implant'].unique()))
print(merged_table['Age at Implant'].median())
print(merged_table['Age at Implant'].min())
print(merged_table['Age at Implant'].max())



49
33.6
16.9
61.6


## Look for lateralization

In [59]:
spikes_thresh = pd.read_csv('/mnt/leif/littlab/users/aguilac/Interictal_Spike_Analysis/HUMAN/working_feat_extract_code/5-propagation/dataset/complete_dfs/hup_thresholded.csv', index_col= 0)


In [83]:
spikes_thresh['pt_id'] = spikes_thresh['pt_id'].str.replace('3T_MP0', '').str.replace('HUP', '')
spikes_thresh['pt_id'] = spikes_thresh['pt_id'].astype(int)

In [86]:
lateralization_of_interest = []
for pt_id, row in spikes_thresh.groupby(by = 'pt_id'):
    if pt_id in merged_table['pt_id'].unique():
        lateralization_of_interest.append([pt_id, row['lateralization'].unique()])
        

In [89]:
lateralization_df = pd.DataFrame(lateralization_of_interest, columns = ['pt_id', 'lateralization'])

In [93]:
lateralization_df['lateralization'].value_counts()

lateralization
[left]         28
[right]        16
[bilateral]     7
Name: count, dtype: int64

In [100]:

## load the spike data
MUSC_spikes = pd.read_csv('/mnt/leif/littlab/users/aguilac/Interictal_Spike_Analysis/HUMAN/working_feat_extract_code/5-propagation/dataset/complete_dfs/MUSC_full.csv', index_col=0)

#load SOZ corrections
MUSC_sozs = pd.read_excel('/mnt/leif/littlab/users/aguilac/Projects/FC_toolbox/results/mat_output_v2/pt_data/MUSC-soz-corrections.xlsx')
MUSC_sozs = MUSC_sozs[MUSC_sozs['Site_1MUSC_2Emory'] == 1]
MUSC_sozs = MUSC_sozs.drop(columns=['Unnamed: 10','Unnamed: 11','Unnamed: 12','Unnamed: 13','Unnamed: 14'])

#fix SOZ and laterality
MUSC_spikes = MUSC_spikes.merge(MUSC_sozs, left_on = 'pt_id', right_on = 'ParticipantID', how = 'inner')
MUSC_spikes = MUSC_spikes.drop(columns=['ParticipantID','Site_1MUSC_2Emory','IfNeocortical_Location','Correction Notes','lateralization_left','lateralization_right','region'])

#find the patients that should be null, and remove them for the full dataset
nonnan_mask = MUSC_sozs.dropna()
pts_to_remove = nonnan_mask[nonnan_mask['Correction Notes'].str.contains('null')]['ParticipantID'].array
MUSC_spikes = MUSC_spikes[~MUSC_spikes['pt_id'].isin(pts_to_remove)]
MUSC_full = MUSC_spikes

## load the spike data
MUSC_spikes = pd.read_csv('/mnt/leif/littlab/users/aguilac/Interictal_Spike_Analysis/HUMAN/working_feat_extract_code/5-propagation/dataset/complete_dfs/MUSC_thresholded.csv', index_col=0)

#load SOZ corrections
MUSC_sozs = pd.read_excel('/mnt/leif/littlab/users/aguilac/Projects/FC_toolbox/results/mat_output_v2/pt_data/MUSC-soz-corrections.xlsx')
MUSC_sozs = MUSC_sozs[MUSC_sozs['Site_1MUSC_2Emory'] == 1]
MUSC_sozs = MUSC_sozs.drop(columns=['Unnamed: 10','Unnamed: 11','Unnamed: 12','Unnamed: 13','Unnamed: 14'])

#fix SOZ and laterality
MUSC_spikes = MUSC_spikes.merge(MUSC_sozs, left_on = 'pt_id', right_on = 'ParticipantID', how = 'inner')
MUSC_spikes = MUSC_spikes.drop(columns=['ParticipantID','Site_1MUSC_2Emory','IfNeocortical_Location','Correction Notes','lateralization_left','lateralization_right','region'])

#remove the patients that should be NULL for the thresholded dataset
MUSC_spikes = MUSC_spikes[~MUSC_spikes['pt_id'].isin(pts_to_remove)]
MUSC_thresh = MUSC_spikes

In [104]:
def assign_lateralization(row):
    if (row['Left'] == 1) & (row['Right'] == 0):
        return 'L'
    elif (row['Left'] == 0) & (row['Right'] == 1):
        return 'R'
    elif (row['Left'] == 1) & (row['Right'] == 1):
        return 'Bi'
    else:
        return None
    
MUSC_thresh['lateralization'] = MUSC_thresh.apply(assign_lateralization, axis=1)

MUSC_thresh['pt_id'] = MUSC_thresh['pt_id'].str.replace('3T_MP0', '').str.replace('HUP', '')
MUSC_thresh['pt_id'] = MUSC_thresh['pt_id'].astype(int)

In [115]:
pt_list['pt_id']

0      15
1      20
2       1
3       3
4       4
     ... 
70    189
71    207
72    209
73    215
74    225
Name: pt_id, Length: 75, dtype: int64

In [116]:
lateralization_of_interest = []
for pt_id, row in MUSC_thresh.groupby(by = 'pt_id'):
    if pt_id in pt_list['pt_id'].unique():
        lateralization_of_interest.append([pt_id, row['lateralization'].unique()])

lateralization_df = pd.DataFrame(lateralization_of_interest, columns = ['pt_id', 'lateralization'])

lateralization_df['lateralization'].value_counts()

lateralization
[Bi]    10
[L]      8
[R]      6
Name: count, dtype: int64

## Surgery performed HUP

In [124]:
merged_table.columns

Index(['HUP Number', 'Type of surgical procedure',
       'Resection laterality:  left or right', 'Resection lobe:',
       'Ablation target:', 'Specify other ablation target:',
       'Months since surgery at time of follow up #1',
       'Follow Up #1 Status (9-15 months from surgery):',
       'ILAE outcome category (past 12 months) for resection and laser patients:',
       'Engel Outcome Classification for resection and laser patients ',
       'Months since surgery at time of follow up #2',
       'ILAE outcome category (past 12 months) for resection and laser patients:.1',
       'Engel Outcome Classification for resection and laser patients .1',
       'Sex assigned at birth', 'MRI Lesion Type:', 'Other (MRI lesion type):',
       'Age at seizure onset?',
       'Age of seizure onset if no age number provided (e.g. early childhood):',
       'Age at Implant', 'Pre-implant hypotheses:', 'spike_rate_corr', 'SOZ',
       'pt_id', 'rise_amp_corr', 'decay_amp_corr', 'sharpness_corr'

In [176]:
print(merged_table['Type of surgical procedure'].value_counts())
print('----------')
print(merged_table['Intervention'].value_counts().divide(51))


def determine_procedure(row):
    if pd.notna(row['Ablation target:']):
        return 'ablated'
    elif pd.notna(row['Resection lobe:']):
        return 'resected'
    else:
        return 'neither'
        
merged_table['ablate_or_resect'] = merged_table.apply(determine_procedure, axis=1)
print('--------')
print(merged_table['ablate_or_resect'].value_counts())

Type of surgical procedure
resection or laser                                      39
neuromodulation (VNS, RNS, DBS)                          8
no surgical intervention received                        3
resection or laser,no surgical intervention received     1
Name: count, dtype: int64
----------
Intervention
Laser Ablation              0.568627
Resection                   0.215686
RNS                         0.117647
No intervention received    0.058824
DBS                         0.039216
Name: count, dtype: float64
--------
ablate_or_resect
ablated     29
neither     13
resected     9
Name: count, dtype: int64


## Outcomes (HUP)

In [177]:
#NEW outcomes - look through them
pec_outcomes = pd.read_csv('/mnt/leif/littlab/users/aguilac/Projects/FC_toolbox/results/mat_output_v2/pt_data/PEC_outcomes.csv')
pec_outcomes = pec_outcomes.dropna(subset='HUP Number')
pec_outcomes = pec_outcomes.drop(columns = 'Follow Up #1 Status (9-15 months from surgery):')
pec_outcomes.columns = ['rid', 'hup_id', 'procedure','resection_laterality','resection_target','ablation_target',
                        'ablation_specific_target','months_f1','ilae_f1','engel_f1',
                        'months_f2','ilae_f2','engel_f2']

pec_outcomes = pec_outcomes[pec_outcomes['procedure'] == 'resection or laser']
pec_outcomes = pec_outcomes.dropna(subset=['ablation_target'])
other_tokeep = ['amygdala and hippocampus','laser ablation of left temporal lobe, hippocampus, and amygdala','hippocampus',
                'Right laser thermal ablation of the hippocampus and amygdala','left hippocampal ablation','amygdala',
                'left Planum Polare and Amygdala (partial)', 'hippocampal', 'left hippocampal ablation and amygdala cyst biopsy',
                'Right Hippocampus', 'left amygdala-hippocampal ablation', 'left parahippocampal focal cortical dysplasia']

pec_outcomes = pec_outcomes[(pec_outcomes['ablation_specific_target'].isin(other_tokeep)) | (pec_outcomes['ablation_target'] == 'Mesial Temporal')]

#load in the outcome data
redcap = pd.read_excel('/mnt/leif/littlab/users/aguilac/Projects/FC_toolbox/results/mat_output_v2/pt_data/Erin_Carlos_RedCAP_data.xlsx')

#create our 2 search queries. We want to look down LOCATION and SURGERY NOTES to get Mesial Temporal targets
outcomes = redcap[~redcap['Location?'].isna()]
outcomes_2 = redcap[~redcap['Surgery NOTES'].isna()]

#grab only mesial temporal structure targetted interventions
outcomes = outcomes[outcomes['Location?'].str.contains('Mesial|mesial|Hippo|hippo|amygd|Amygd')]
outcomes = outcomes[~outcomes['Procedure?'].str.contains('Resection|resection')]
outcomes = outcomes[outcomes['Location?'].str.contains('Mesial Temporal')]

#do the same but across outcomes_2
# outcomes_2 = outcomes_2[outcomes_2['Surgery NOTES'].str.contains('Mesial|mesial|Hippo|hippo|amygd|Amygd')]

#now merge them to see what we have
# mesial_pts = pd.concat([outcomes, outcomes_2])
mesial_pts = outcomes
mesial_pts = mesial_pts.drop_duplicates().reset_index(drop = True)
mesial_pts = mesial_pts[~mesial_pts['Outcomes?'].isna()]
mesial_pts = mesial_pts[~mesial_pts['Outcomes?'].str.contains('NONE|OTHER|None')]

#seperate between good and bad
split_outcomes = mesial_pts['Outcomes?'].str.split()
ilae_indices = split_outcomes.apply(lambda x: x[-1] for x in split_outcomes)
mesial_pts['ilae'] = ilae_indices.iloc[:, 1]

def map_ilae_to_go(ilae_value, which):
    if ilae_value in which:
        return 1
    else:
        return 0
    
which = ['1','1a']
mesial_pts['G/O v1'] = mesial_pts['ilae'].apply(lambda x: map_ilae_to_go(x, which))
which = ['1','2','1a']
mesial_pts['G/O v2'] = mesial_pts['ilae'].apply(lambda x: map_ilae_to_go(x, which))

pts_oi = mesial_pts[['HUP_id','G/O v1','G/O v2']]
pts_oi['HUP_id'] = pts_oi['HUP_id'].str.replace('3T_MP0', '').str.replace('HUP', '')
pts_oi = pts_oi.rename(columns = {'HUP_id':'hup_id'})

pts_oi['hup_id'] = pts_oi['hup_id'].astype(int)
pec_outcomes['hup_id'] = pec_outcomes['hup_id'].astype(int)

pec_outcomes = pec_outcomes.merge(pts_oi, on= 'hup_id', how = 'left')


/tmp/ipykernel_50085/1868566447.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pts_oi['HUP_id'] = pts_oi['HUP_id'].str.replace('3T_MP0', '').str.replace('HUP', '')


In [180]:
all_outcomes = pec_outcomes.merge(merged_table[['pt_id']], left_on='hup_id', right_on='pt_id', how = 'inner')

In [181]:
all_outcomes

,rid,hup_id,procedure,resection_laterality,resection_target,ablation_target,ablation_specific_target,months_f1,ilae_f1,engel_f1,months_f2,ilae_f2,engel_f2,G/O v1,G/O v2,pt_id
0,89,126,resection or laser,NaN,NaN,Mesial Temporal,NaN,14.0,"Seizure free since surgery, no auras",IA: Completely seizure-free since surgery <br>,24.0,"Seizure free since surgery, no auras",IA: Completely seizure-free since surgery <br>,1.0,1.0,126
1,274,135,resection or laser,NaN,NaN,Other (specify),amygdala and hippocampus,11.0,Rare seizures (1-3 seizure days per year),IIB: Rare disabling seizures since surgery,NaN,Rare seizures (1-3 seizure days per year),IIB: Rare disabling seizures since surgery,0.0,0.0,135
2,278,138,resection or laser,NaN,NaN,Other (specify),"laser ablation of left temporal lobe, hippocam...",NaN,Rare seizures (1-3 seizure days per year),IIA: Initially free of disabling seizures but...,23.0,Seizure reduction >50% (but >3 seizure days/year),IVA: Significant seizure reduction,NaN,NaN,138
3,320,140,resection or laser,NaN,NaN,Other (specify),hippocampus,NaN,"Auras only, no other seizures",IB: Non disabling simple partial seizures onl...,27.0,"Auras only, no other seizures",IB: Non disabling simple partial seizures onl...,0.0,1.0,140
4,294,141,resection or laser,NaN,NaN,Other (specify),Right laser thermal ablation of the hippocampu...,NaN,Rare seizures (1-3 seizure days per year),IIB: Rare disabling seizures since surgery,NaN,"Seizure free since surgery, no auras","IC: Some disabling seizures after surgery, bu...",1.0,1.0,141
5,295,142,resection or laser,NaN,NaN,Other (specify),left hippocampal ablation,NaN,Seizure reduction >50% (but >3 seizure days/year),IIIA: Worthwhile seizure reduction,NaN,Seizure reduction >50% (but >3 seizure days/year),IIIA: Worthwhile seizure reduction,0.0,1.0,142
6,338,161,resection or laser,NaN,NaN,Mesial Temporal,NaN,32.0,No change (between 50% seizure reduction and 1...,IVB: No appreciable change,NaN,No change (between 50% seizure reduction and 1...,IVB: No appreciable change,0.0,0.0,161
7,412,162,resection or laser,NaN,NaN,Mesial Temporal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,162
8,279,163,resection or laser,NaN,NaN,Mesial Temporal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,163
9,325,165,resection or laser,NaN,NaN,Mesial Temporal,NaN,NaN,Rare seizures (1-3 seizure days per year),IIB: Rare disabling seizures since surgery,29.0,Rare seizures (1-3 seizure days per year),IIB: Rare disabling seizures since surgery,0.0,0.0,165
